# Scrabble AI on Colab

Upload this notebook to Colab and choose **Runtime → Run all** on a CPU runtime. The notebook contains the demo source and its 51,742-word dictionary. The first run builds a cache in the runtime.

The first move is a quick check. The AI should play **QUIZ for 44 points** from `Q U I Z B C G`. Use **Play AI move** below to advance one turn at a time.

In [ ]:
# @title Set up the bundled Scrabble demo
from base64 import b64decode
from importlib.util import find_spec
from io import BytesIO
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import subprocess
import sys

missing = [name for name in ("numpy", "ipywidgets") if find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

if "scrabble_demo_folder" not in globals():
    scrabble_demo_folder = TemporaryDirectory(prefix="scrabble-colab-")
source_directory = Path(scrabble_demo_folder.name)
bundle = (
    '...'
)
with ZipFile(BytesIO(b64decode(bundle))) as archive:
    archive.extractall(source_directory)
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))
print("Scrabble demo files are ready.")

## Verify the opening

At depth 1 the AI ranks moves by their immediate score. The four tiles in QUIZ add up to 22 points, and the center square doubles the word score. The exact starting square can vary among tied plays.

In [ ]:
from demo import DemoSession, render_ascii

session = DemoSession()
session.configure_search(depth=1, width=1, candidates="1", random_opponent_rack=False)
print("Loading the dictionary...", flush=True)
session.load_engine()
session.play_move()

assert session.game.players[0]["score"] == 44
assert "QUIZ  +44 points" in session.last_move
assert (7, 7) in session.new_tiles
print(session.last_move)

session.reset()

## Play the demo

Search settings apply to the next move. Enter comma-separated candidate counts such as `3,2,1`. A short list repeats its last count. At depth 1, width and rack sampling do not affect the move ranking.

Try **Reset**, set **Candidates** to `3`, and play again at depth 1. Compare the word and score with the opening check above.

If widgets are unavailable, run `session.play_move(); print(render_ascii(session))` in a new code cell to take a turn in text.

In [ ]:
import asyncio
from html import escape

import ipywidgets as widgets
from IPython.display import display


def make_demo_panel(session):
    settings = session.search_settings
    style = {"description_width": "initial"}
    depth = widgets.BoundedIntText(
        value=settings.depth, min=1, max=4, description="Depth",
        style=style, layout=widgets.Layout(width="125px"),
    )
    width = widgets.BoundedIntText(
        value=settings.width, min=1, max=10, description="Width",
        style=style, layout=widgets.Layout(width="125px"),
    )
    candidates = widgets.Text(
        value=",".join(map(str, settings.candidates)), description="Candidates",
        style=style, layout=widgets.Layout(width="185px"),
    )
    sample_rack = widgets.Checkbox(
        value=settings.random_opponent_rack, description="Random rack sampling",
        indent=False,
    )
    play = widgets.Button(description="Play AI move", button_style="primary")
    reset = widgets.Button(description="Reset")
    status = widgets.HTML()
    board = widgets.HTML()
    active_turn = None

    def redraw():
        text = escape(render_ascii(session))
        board.value = (
            '<pre style="overflow-x:auto;padding:12px;'
            'font-family:IBM Plex Mono,Consolas,monospace;line-height:1.35;">'
            f'{text}</pre>'
        )

    async def finish_turn():
        try:
            await asyncio.to_thread(session.play_move)
        except Exception as error:
            status.value = escape(f"Search failed. {error}")
        else:
            status.value = "Reset to play again." if session.finished else ""
        finally:
            play.disabled = session.finished
            reset.disabled = False
            redraw()

    def start_turn(_):
        nonlocal active_turn
        if play.disabled or (active_turn is not None and not active_turn.done()):
            return
        try:
            session.configure_search(depth.value, width.value, candidates.value, sample_rack.value)
        except ValueError as error:
            status.value = escape(str(error))
            return
        play.disabled = reset.disabled = True
        status.value = f"Player {session.game.current_player + 1} is thinking…"
        active_turn = asyncio.create_task(finish_turn())

    def reset_game(_):
        if reset.disabled:
            return
        session.reset()
        play.disabled = False
        status.value = ""
        redraw()

    play.on_click(start_turn)
    reset.on_click(reset_game)
    redraw()
    controls = widgets.Box(
        [depth, width, candidates],
        layout=widgets.Layout(display="flex", flex_flow="row wrap", gap="8px"),
    )
    return widgets.VBox([
        controls,
        sample_rack,
        widgets.HBox([play, reset]),
        status,
        board,
    ])


panel = make_demo_panel(session)
display(panel)